In [29]:
# Install required libraries for PDF generation and Jaal Diagram
!pip install reportlab jaal

3737.05s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


You should consider upgrading via the '/home/opc/DevOps/dep-diag/venv/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/home/opc/DevOps/dep-diag/venv/bin/python3 -m pip install --upgrade pip' command.


In [30]:
# Enhanced Infrastructure & Application Stack Analysis
# Correlates inventory.csv (server inventory) with strat-dump.csv (network dependencies)

import numpy as np
from collections import defaultdict

# Update file paths to new names
print("🔄 Loading updated datasets...")

# Load Stratozone network dump
strat_file = 'strat-dump.csv'
try:
    strat_df = pd.read_csv(strat_file)
    print(f"✅ Loaded Stratozone dump: {strat_df.shape[0]} connections")
except FileNotFoundError:
    print("❌ strat-dump.csv not found. Using existing df if available.")
    strat_df = df if 'df' in globals() else None

# Load full inventory
inventory_file = 'inventory.csv'
try:
    # Use robust CSV parsing to handle malformed data
    import csv
    
    print("🔧 Parsing inventory CSV with error handling...")
    rows = []
    with open(inventory_file, 'r', encoding='utf-8-sig') as f:
        reader = csv.reader(f)
        header = next(reader)
        
        fixed_rows = 0
        for i, row in enumerate(reader, 1):
            if len(row) > len(header):
                # Truncate extra fields
                row = row[:len(header)]
                fixed_rows += 1
            elif len(row) < len(header):
                # Pad with empty strings
                row.extend([''] * (len(header) - len(row)))
                fixed_rows += 1
            rows.append(row)
    
    # Create DataFrame
    inventory_df = pd.DataFrame(rows, columns=header)
    
    # Clean up column names (remove BOM characters)
    inventory_df.columns = [col.strip().replace('\ufeff', '') for col in inventory_df.columns]
    
    print(f"✅ Loaded inventory: {inventory_df.shape[0]} servers")
    if fixed_rows > 0:
        print(f"⚠️ Fixed {fixed_rows} malformed rows")
    
    # Display inventory structure
    print("\n=== Inventory Data Structure ===")
    print("Columns:", list(inventory_df.columns))
    print("\nFirst few rows:")
    display(inventory_df.head())
    
    # Key statistics
    print(f"\n📊 Inventory Statistics:")
    print(f"- Unique devices: {inventory_df['name'].nunique()}")
    print(f"- Unique IPs: {inventory_df['collectedIpAddress'].nunique()}")
    if 'roleName' in inventory_df.columns:
        print(f"- Role distribution: {inventory_df['roleName'].value_counts().to_dict()}")
    if 'osName' in inventory_df.columns:
        print(f"- OS distribution: {inventory_df['osName'].value_counts().head().to_dict()}")
    
except FileNotFoundError:
    print("❌ inventory.csv not found. Please ensure the file is in the same directory.")
    inventory_df = None
except Exception as e:
    print(f"❌ Error loading inventory: {e}")
    inventory_df = None

if inventory_df is not None and strat_df is not None:
    print("\n🔗 Starting Enhanced Correlation Analysis...")
    
    # === DATA PREPARATION ===
    print("\n=== 1. Data Preparation & Normalization ===")
    
    # Normalize device names for better matching
    inventory_df['name_clean'] = inventory_df['name'].str.upper().str.strip()
    strat_df['initiating_clean'] = strat_df['initiatingDeviceName'].str.upper().str.strip()
    strat_df['receiving_clean'] = strat_df['receivingDeviceName'].str.upper().str.strip()
    
    # Create mapping dictionaries
    # Handle duplicate device names by keeping the first occurrence
    inventory_unique_names = inventory_df.drop_duplicates(subset=['name_clean'], keep='first')
    inventory_unique_ips = inventory_df.drop_duplicates(subset=['collectedIpAddress'], keep='first') if 'collectedIpAddress' in inventory_df.columns else inventory_df
    
    name_to_inventory = inventory_unique_names.set_index('name_clean').to_dict('index')
    ip_to_inventory = inventory_unique_ips.set_index('collectedIpAddress').to_dict('index') if 'collectedIpAddress' in inventory_df.columns else {}
    
    print(f"📋 Created mappings:")
    print(f"  - Name-based: {len(name_to_inventory)} entries")
    print(f"  - IP-based: {len(ip_to_inventory)} entries")
    
    # === CORRELATION ENGINE ===
    print("\n=== 2. Multi-Method Correlation Engine ===")
    
    def enrich_device_info(device_name, device_ip, suffix=""):
        """Enrich device with inventory information"""
        info = {}
        
        # Try name-based lookup first
        if device_name in name_to_inventory:
            source_info = name_to_inventory[device_name]
            info.update({f"{col}{suffix}": source_info.get(col, 'Unknown') 
                        for col in ['roleName', 'osName', 'manufacturer', 'model', 
                                   'cpuType', 'memoryInMb', 'isLinux', 'isPhysical'] 
                        if col in source_info})
            info[f'match_method{suffix}'] = 'name'
        
        # Fall back to IP-based lookup
        elif device_ip in ip_to_inventory:
            source_info = ip_to_inventory[device_ip]
            info.update({f"{col}{suffix}": source_info.get(col, 'Unknown') 
                        for col in ['roleName', 'osName', 'manufacturer', 'model', 
                                   'cpuType', 'memoryInMb', 'isLinux', 'isPhysical'] 
                        if col in source_info})
            info[f'match_method{suffix}'] = 'ip'
        
        else:
            # Default values for unmatched devices
            info.update({
                f'roleName{suffix}': 'Unknown',
                f'osName{suffix}': 'Unknown',
                f'manufacturer{suffix}': 'Unknown',
                f'match_method{suffix}': 'none'
            })
        
        return info
    
    # Apply enrichment to all connections
    enriched_connections = []
    for _, row in strat_df.iterrows():
        enriched_row = row.to_dict()
        
        # Enrich initiating device
        initiating_info = enrich_device_info(
            row['initiating_clean'], 
            row['initiatingIpAddress'], 
            '_initiating'
        )
        enriched_row.update(initiating_info)
        
        # Enrich receiving device
        receiving_info = enrich_device_info(
            row['receiving_clean'], 
            row['receivingIpAddress'], 
            '_receiving'
        )
        enriched_row.update(receiving_info)
        
        enriched_connections.append(enriched_row)
    
    # Create enriched DataFrame
    enriched_df = pd.DataFrame(enriched_connections)
    
    print(f"✅ Enrichment complete:")
    print(f"  - Total connections: {len(enriched_df)}")
    print(f"  - Initiating matches: {(enriched_df['match_method_initiating'] != 'none').sum()}")
    print(f"  - Receiving matches: {(enriched_df['match_method_receiving'] != 'none').sum()}")
    
    # === APPLICATION STACK INFERENCE ===
    print("\n=== 3. Application Stack Inference ===")
    
    def infer_application_stack(device_name, device_role, os_name, connections_in, connections_out, ports_used):
        """Infer complete application stack for a device"""
        stack = {
            'primary_role': device_role,
            'os_platform': os_name,
            'application_tier': 'Unknown',
            'technology_stack': [],
            'database_engines': [],
            'web_technologies': [],
            'middleware': [],
            'confidence_score': 0
        }
        
        # Role-based inference
        if 'database' in str(device_role).lower():
            stack['application_tier'] = 'Database Tier'
            # Analyze database ports
            if 1521 in ports_used or 2483 in ports_used:
                stack['database_engines'].append('Oracle Database')
                stack['confidence_score'] += 30
            if 3306 in ports_used:
                stack['database_engines'].append('MySQL')
                stack['confidence_score'] += 30
            if 5432 in ports_used:
                stack['database_engines'].append('PostgreSQL')
                stack['confidence_score'] += 30
            if 1433 in ports_used:
                stack['database_engines'].append('SQL Server')
                stack['confidence_score'] += 30
                
        elif 'web' in str(device_role).lower():
            stack['application_tier'] = 'Web Tier'
            if 80 in ports_used or 8080 in ports_used:
                stack['web_technologies'].append('HTTP Server')
                stack['confidence_score'] += 20
            if 443 in ports_used or 8443 in ports_used:
                stack['web_technologies'].append('HTTPS Server')
                stack['confidence_score'] += 20
                
        elif 'application' in str(device_role).lower():
            stack['application_tier'] = 'Application Tier'
            stack['confidence_score'] += 15
            
        # OS-based technology inference
        if 'linux' in str(os_name).lower():
            stack['technology_stack'].extend(['Linux', 'Open Source Stack'])
        elif 'windows' in str(os_name).lower():
            stack['technology_stack'].extend(['Windows', 'Microsoft Stack'])
            
        # Name-based inference
        device_name_lower = str(device_name).lower()
        if 'oracle' in device_name_lower:
            stack['database_engines'].append('Oracle Database')
            stack['confidence_score'] += 25
        if 'mysql' in device_name_lower:
            stack['database_engines'].append('MySQL')
            stack['confidence_score'] += 25
        if 'web' in device_name_lower or 'apache' in device_name_lower:
            stack['web_technologies'].append('Web Server')
            stack['confidence_score'] += 20
            
        return stack
    
    # Analyze each unique device
    device_stacks = {}
    
    print("🔍 Analyzing application stacks...")
    
    # Get unique devices from both initiating and receiving
    all_devices = set()
    all_devices.update(zip(enriched_df['initiatingDeviceName'], enriched_df['roleName_initiating'], enriched_df['osName_initiating']))
    all_devices.update(zip(enriched_df['receivingDeviceName'], enriched_df['roleName_receiving'], enriched_df['osName_receiving']))
    
    for device_name, role, os_name in all_devices:
        if pd.isna(device_name):
            continue
            
        # Get connection patterns for this device
        device_connections_out = enriched_df[enriched_df['initiatingDeviceName'] == device_name]
        device_connections_in = enriched_df[enriched_df['receivingDeviceName'] == device_name]
        
        # Get ports used by this device
        # Handle non-numeric port values safely
        try:
            ports_out_series = device_connections_out['initiatingPort'].dropna()
            ports_out = set()
            for port in ports_out_series:
                try:
                    if str(port).isdigit():
                        ports_out.add(int(port))
                except:
                    continue
            
            ports_in_series = device_connections_in['receivingPort'].dropna()
            ports_in = set()
            for port in ports_in_series:
                try:
                    if str(port).isdigit():
                        ports_in.add(int(port))
                except:
                    continue
                    
            all_ports = ports_out.union(ports_in)
        except Exception as e:
            print(f"Warning: Port parsing error for {device_name}: {e}")
            all_ports = set()
        
        # Infer application stack
        stack = infer_application_stack(
            device_name, role, os_name, 
            len(device_connections_in), len(device_connections_out), 
            all_ports
        )
        
        device_stacks[device_name] = stack
    
    print(f"✅ Analyzed {len(device_stacks)} unique devices")
    
    # === COMPREHENSIVE ANALYSIS ===
    print("\n=== 4. Infrastructure Analysis Results ===")
    
    # Application Tier Distribution
    print("\n📊 Application Tier Distribution:")
    tier_counts = defaultdict(int)
    for stack in device_stacks.values():
        tier_counts[stack['application_tier']] += 1
    
    for tier, count in sorted(tier_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {tier}: {count} devices")
    
    # Technology Stack Analysis
    print("\n🔧 Technology Stack Analysis:")
    tech_counts = defaultdict(int)
    for stack in device_stacks.values():
        for tech in stack['technology_stack']:
            tech_counts[tech] += 1
    
    for tech, count in sorted(tech_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {tech}: {count} devices")
    
    # Database Engine Distribution
    print("\n🗄️ Database Engine Distribution:")
    db_counts = defaultdict(int)
    for stack in device_stacks.values():
        for db in stack['database_engines']:
            db_counts[db] += 1
    
    for db, count in sorted(db_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {db}: {count} devices")
    
    # === DETAILED DEVICE ANALYSIS ===
    print("\n=== 5. Detailed Device Analysis ===")
    
    # Create comprehensive device summary
    device_summary = []
    for device_name, stack in device_stacks.items():
        summary = {
            'Device': device_name,
            'Role': stack['primary_role'],
            'OS': stack['os_platform'],
            'Tier': stack['application_tier'],
            'Technologies': ', '.join(stack['technology_stack']) if stack['technology_stack'] else 'Unknown',
            'Databases': ', '.join(stack['database_engines']) if stack['database_engines'] else 'None',
            'Web_Tech': ', '.join(stack['web_technologies']) if stack['web_technologies'] else 'None',
            'Confidence': stack['confidence_score']
        }
        device_summary.append(summary)
    
    device_summary_df = pd.DataFrame(device_summary)
    device_summary_df = device_summary_df.sort_values('Confidence', ascending=False)
    
    print("\n🎯 Top 20 Most Confident Application Stack Inferences:")
    display(device_summary_df.head(20))
    
    # === CONNECTION FLOW ANALYSIS ===
    print("\n=== 6. Application Flow Analysis ===")
    
    # Analyze tier-to-tier communication
    tier_flows = enriched_df.groupby(['roleName_initiating', 'roleName_receiving']).agg({
        'connectionCount': 'sum',
        'initiatingDeviceName': 'nunique',
        'receivingDeviceName': 'nunique'
    }).rename(columns={
        'connectionCount': 'Total_Connections',
        'initiatingDeviceName': 'Unique_Sources',
        'receivingDeviceName': 'Unique_Targets'
    }).reset_index().sort_values('Total_Connections', ascending=False)
    
    print("\n🔄 Top Application Flow Patterns:")
    display(tier_flows.head(15))
    
    # === CRITICAL SYSTEMS IDENTIFICATION ===
    print("\n=== 7. Critical Systems Identification ===")
    
    # Identify systems with highest connectivity
    device_connectivity = {}
    for device in device_stacks.keys():
        incoming = enriched_df[enriched_df['receivingDeviceName'] == device]['connectionCount'].sum()
        outgoing = enriched_df[enriched_df['initiatingDeviceName'] == device]['connectionCount'].sum()
        unique_connections = len(set(
            enriched_df[enriched_df['receivingDeviceName'] == device]['initiatingDeviceName'].tolist() +
            enriched_df[enriched_df['initiatingDeviceName'] == device]['receivingDeviceName'].tolist()
        ))
        
        device_connectivity[device] = {
            'incoming_connections': incoming,
            'outgoing_connections': outgoing,
            'total_connections': incoming + outgoing,
            'unique_peers': unique_connections,
            'stack': device_stacks[device]
        }
    
    # Sort by total connections to identify critical systems
    critical_systems = sorted(device_connectivity.items(), 
                            key=lambda x: x[1]['total_connections'], reverse=True)
    
    print("\n🔥 Top 15 Critical Systems (by connection volume):")
    critical_df_data = []
    for device, stats in critical_systems[:15]:
        critical_df_data.append({
            'Device': device,
            'Role': stats['stack']['primary_role'],
            'Total_Connections': stats['total_connections'],
            'Incoming': stats['incoming_connections'],
            'Outgoing': stats['outgoing_connections'],
            'Unique_Peers': stats['unique_peers'],
            'App_Tier': stats['stack']['application_tier'],
            'Confidence': stats['stack']['confidence_score']
        })
    
    critical_df = pd.DataFrame(critical_df_data)
    display(critical_df)
    
    # Store results for further analysis
    globals()['enriched_network_df'] = enriched_df
    globals()['device_stacks'] = device_stacks
    globals()['device_summary_df'] = device_summary_df
    globals()['critical_systems_df'] = critical_df
    
    print(f"\n✅ Analysis Complete! Data stored in global variables:")
    print(f"  - enriched_network_df: {enriched_df.shape[0]} enriched connections")
    print(f"  - device_stacks: {len(device_stacks)} device application stacks")
    print(f"  - device_summary_df: Device summary table")
    print(f"  - critical_systems_df: Critical systems analysis")
    
else:
    print("❌ Cannot proceed with analysis. Please ensure both inventory.csv and strat-dump.csv are available.")

🔄 Loading updated datasets...
✅ Loaded Stratozone dump: 19629 connections
🔧 Parsing inventory CSV with error handling...
✅ Loaded inventory: 385 servers
⚠️ Fixed 385 malformed rows

=== Inventory Data Structure ===
Columns: ['groupType', 'groupName', 'solutionComponent', 'name', 'domain', 'createDate', 'collectedIpAddress', 'manufacturer', 'model', 'isLinux', 'isPhysical', 'ageInMonths', 'collectedFromLocation', 'collectionMethod', 'osName', 'cloudFitScore', 'cpuType', 'cpuArchitecture', 'roleName', 'organizationalUnitName', 'processorCount', 'totalProcessorCores', 'totalProcessorThreads', 'memoryInMb', 'driveTotalCapacityInGb', 'driveTotalFreeInGb', 'driveTotalUsedInGb', 'estimatedMonthlyDataReceivedInGb', 'estimatedMonthlyDataSentInGb', 'vmwareVCenterUrl', 'vmwareVmId', 'vmwareVmName', 'vmwareVmPowerState', 'upSince', 'lastCollectedDate']

First few rows:
✅ Loaded Stratozone dump: 19629 connections
🔧 Parsing inventory CSV with error handling...
✅ Loaded inventory: 385 servers
⚠️ Fixe

,groupType,groupName,solutionComponent,name,domain,createDate,collectedIpAddress,manufacturer,model,isLinux,...,driveTotalFreeInGb,driveTotalUsedInGb,estimatedMonthlyDataReceivedInGb,estimatedMonthlyDataSentInGb,vmwareVCenterUrl,vmwareVmId,vmwareVmName,vmwareVmPowerState,upSince,lastCollectedDate
0,Assessment,MP,,LNXADMDEV01,afphabitat.cl,30-Apr-2021,192.168.130.39,Unknown,Unknown,Yes,...,94,35,26,101,68,605,528,,,
1,Assessment,MP,,LNXADMDEV01,,30-Apr-2021,192.168.130.39,Unknown,Unknown,Yes,...,00,30,00,70,00,597,523,,,
2,Assessment,MP,,CYB-PVWA02,afphabitat.cl,02-May-2021,192.168.200.175,VMware,Inc.,VMware7,...,8.191,79,45,53,74,25,70,20,18,
3,Assessment,MP,,APPSDESA,afphabitat.cl,02-May-2021,192.168.10.49,Unknown,Unknown,Yes,...,74,10,47,7,27,0,0,,,
4,Assessment,MP,,LNXFEPROD01,afphabitat.cl,02-May-2021,192.168.200.31,Unknown,Unknown,Yes,...,94,190,37,183,57,707,2.711,,,



📊 Inventory Statistics:
- Unique devices: 275
- Unique IPs: 274
- Role distribution: {'x64': 130, 'Unassigned': 60, 'Unknown': 54, 'Middleware': 40, 'Application Server': 31, 'Web Server': 25, 'Database Server': 20, 'Intel Celeron processor': 8, 'Intel Xeon Silver 4210 @ 2.20GHz': 4, 'Intel Xeon E5-2620 @ 2.00GHz': 4, 'Intel Xeon Gold 6148 @ 2.40GHz': 4, 'Infrastructure Server': 2, 'Intel Xeon E5-2640 v4 @ 2.40GHz': 2, 'Intel Xeon E5-2650 @ 2.00GHz': 1}
- OS distribution: {'StratoProbe Guest OS Scan': 196, 'Red Hat Enterprise Linux Server 5.8': 61, 'Other Linux': 42, 'Red Hat Enterprise Linux Server 5.5': 19, 'MP': 11}

🔗 Starting Enhanced Correlation Analysis...

=== 1. Data Preparation & Normalization ===
📋 Created mappings:
  - Name-based: 275 entries
  - IP-based: 274 entries

=== 2. Multi-Method Correlation Engine ===
✅ Enrichment complete:
  - Total connections: 19629
  - Initiating matches: 19629
  - Receiving matches: 19629

=== 3. Application Stack Inference ===
🔍 Analyzing a

,Device,Role,OS,Tier,Technologies,Databases,Web_Tech,Confidence
69,MYSQLJOOMLA1,Database Server,Red Hat Enterprise Linux Server 5.7,Database Tier,"Linux, Open Source Stack","MySQL, MySQL",None,55
34,VCERMYSQLJOOMLA1,Database Server,Other Linux,Database Tier,"Linux, Open Source Stack","MySQL, MySQL",None,55
204,MYSQLJOOMLA2,Database Server,Red Hat Enterprise Linux Server 5.7,Database Tier,"Linux, Open Source Stack","MySQL, MySQL",None,55
161,DBCERT11G-RAC2,Database Server,Oracle Linux Server 5.9,Database Tier,"Linux, Open Source Stack",Oracle Database,None,30
165,DBCONT11G-RAC2,Database Server,Oracle Linux Server 5.9,Database Tier,"Linux, Open Source Stack",Oracle Database,None,30
167,DBCONT11G-RAC1,Database Server,Other Linux,Database Tier,"Linux, Open Source Stack",Oracle Database,None,30
115,DBCERT11G-RAC1,Database Server,Oracle Linux Server 5.9,Database Tier,"Linux, Open Source Stack",Oracle Database,None,30
241,VDEV-RTC-ALUI01,Web Server,Red Hat Enterprise Linux Server ES 4,Web Tier,"Linux, Open Source Stack",None,HTTP Server,20
73,FTPSVR02,Web Server,Red Hat Enterprise Linux Server 5.5,Web Tier,"Linux, Open Source Stack",None,HTTP Server,20
91,VDEV-RTC-ALUI02,Web Server,Red Hat Enterprise Linux Server ES 4,Web Tier,"Linux, Open Source Stack",None,HTTP Server,20



=== 6. Application Flow Analysis ===

🔄 Top Application Flow Patterns:


,roleName_initiating,roleName_receiving,Total_Connections,Unique_Sources,Unique_Targets
113,x64,x64,77986,101,103
91,Unknown,x64,41493,25,24
7,Application Server,Unassigned,37259,13,17
103,x64,Intel Celeron processor,28251,53,7
39,Intel Xeon E5-2640 v4 @ 2.40GHz,x64,26497,2,7
111,x64,Unknown,24854,20,27
106,x64,Intel Xeon E5-2650 @ 2.00GHz,24380,52,1
72,Middleware,x64,18865,26,17
1,Application Server,Database Server,14326,8,5
10,Application Server,x64,10830,21,17



=== 7. Critical Systems Identification ===

🔥 Top 15 Critical Systems (by connection volume):

🔥 Top 15 Critical Systems (by connection volume):


,Device,Role,Total_Connections,Incoming,Outgoing,Unique_Peers,App_Tier,Confidence
0,LNXDBPROD10G-RAC1,x64,35336,33902,1434,52,Unknown,0
1,VPROD-EX13MBX2,x64,32789,8623,24166,18,Unknown,0
2,NET-DOM03,Intel Xeon E5-2650 @ 2.00GHz,30858,28335,2523,99,Unknown,0
3,LNXSCJAVA01,Unknown,29887,79,29808,9,Unknown,0
4,DBDETE11G-RAC2,Unassigned,24999,24720,279,28,Unknown,0
5,VPROD-EX13MBX1,x64,24495,4024,20471,18,Unknown,0
6,LNXAPPDEV01,Application Server,24121,340,23781,17,Application Tier,15
7,LNXDBPLAN12C,x64,24086,23921,165,18,Unknown,0
8,NET-DOM02,Intel Celeron processor,20934,19500,1434,80,Unknown,0
9,LNXAPPDEV02,Application Server,18211,599,17612,17,Application Tier,15



✅ Analysis Complete! Data stored in global variables:
  - enriched_network_df: 19629 enriched connections
  - device_stacks: 273 device application stacks
  - device_summary_df: Device summary table
  - critical_systems_df: Critical systems analysis


In [ ]:
# Enhanced Export with Comprehensive Technology Stack Analysis
# This creates a detailed CSV file with network connections, inventory data, and inferred tech stacks

import os
from datetime import datetime

print("📤 Preparing comprehensive enriched network data for export...")

# Enhanced port-to-technology mapping for detailed inference
def get_detailed_port_analysis(port_num):
    """Get detailed technology stack analysis for a specific port"""
    
    # Comprehensive port mapping with technology details
    detailed_port_map = {
        # Web Technologies
        80: {'service': 'HTTP', 'category': 'Web', 'technology': 'HTTP Server', 'stack': 'Web Frontend', 'protocol': 'TCP'},
        443: {'service': 'HTTPS', 'category': 'Web', 'technology': 'SSL/TLS Web Server', 'stack': 'Secure Web Frontend', 'protocol': 'TCP'},
        8080: {'service': 'HTTP-Alt', 'category': 'Web', 'technology': 'Alternative HTTP Server', 'stack': 'Web Application', 'protocol': 'TCP'},
        8443: {'service': 'HTTPS-Alt', 'category': 'Web', 'technology': 'Alternative HTTPS Server', 'stack': 'Secure Web Application', 'protocol': 'TCP'},
        8000: {'service': 'HTTP-Dev', 'category': 'Web', 'technology': 'Development Web Server', 'stack': 'Development Environment', 'protocol': 'TCP'},
        3000: {'service': 'Node.js', 'category': 'Web', 'technology': 'Node.js Application', 'stack': 'JavaScript Runtime', 'protocol': 'TCP'},
        4200: {'service': 'Angular', 'category': 'Web', 'technology': 'Angular Development Server', 'stack': 'Frontend Framework', 'protocol': 'TCP'},
        3001: {'service': 'React', 'category': 'Web', 'technology': 'React Development Server', 'stack': 'Frontend Framework', 'protocol': 'TCP'},
        
        # Database Technologies
        1521: {'service': 'Oracle DB', 'category': 'Database', 'technology': 'Oracle Database', 'stack': 'Enterprise Database', 'protocol': 'TCP'},
        3306: {'service': 'MySQL', 'category': 'Database', 'technology': 'MySQL Database', 'stack': 'Relational Database', 'protocol': 'TCP'},
        5432: {'service': 'PostgreSQL', 'category': 'Database', 'technology': 'PostgreSQL Database', 'stack': 'Open Source Database', 'protocol': 'TCP'},
        1433: {'service': 'SQL Server', 'category': 'Database', 'technology': 'Microsoft SQL Server', 'stack': 'Microsoft Database', 'protocol': 'TCP'},
        27017: {'service': 'MongoDB', 'category': 'Database', 'technology': 'MongoDB Database', 'stack': 'NoSQL Database', 'protocol': 'TCP'},
        6379: {'service': 'Redis', 'category': 'Database', 'technology': 'Redis Cache', 'stack': 'In-Memory Database', 'protocol': 'TCP'},
        5984: {'service': 'CouchDB', 'category': 'Database', 'technology': 'Apache CouchDB', 'stack': 'Document Database', 'protocol': 'TCP'},
        7000: {'service': 'Cassandra', 'category': 'Database', 'technology': 'Apache Cassandra', 'stack': 'Distributed Database', 'protocol': 'TCP'},
        
        # Application Servers & Middleware
        8081: {'service': 'Tomcat', 'category': 'Application', 'technology': 'Apache Tomcat', 'stack': 'Java Application Server', 'protocol': 'TCP'},
        9080: {'service': 'WebSphere', 'category': 'Application', 'technology': 'IBM WebSphere', 'stack': 'Enterprise Application Server', 'protocol': 'TCP'},
        7001: {'service': 'WebLogic', 'category': 'Application', 'technology': 'Oracle WebLogic', 'stack': 'Java EE Application Server', 'protocol': 'TCP'},
        8090: {'service': 'JBoss', 'category': 'Application', 'technology': 'JBoss/WildFly', 'stack': 'Java Application Server', 'protocol': 'TCP'},
        9000: {'service': 'Generic App', 'category': 'Application', 'technology': 'Application Server', 'stack': 'Custom Application', 'protocol': 'TCP'},
        
        # Message Queues & Streaming
        5672: {'service': 'RabbitMQ', 'category': 'Messaging', 'technology': 'RabbitMQ', 'stack': 'Message Broker', 'protocol': 'TCP'},
        9092: {'service': 'Kafka', 'category': 'Messaging', 'technology': 'Apache Kafka', 'stack': 'Event Streaming', 'protocol': 'TCP'},
        61616: {'service': 'ActiveMQ', 'category': 'Messaging', 'technology': 'Apache ActiveMQ', 'stack': 'Message Broker', 'protocol': 'TCP'},
        
        # Monitoring & Management
        9200: {'service': 'Elasticsearch', 'category': 'Search', 'technology': 'Elasticsearch', 'stack': 'Search Engine', 'protocol': 'TCP'},
        5601: {'service': 'Kibana', 'category': 'Monitoring', 'technology': 'Kibana Dashboard', 'stack': 'Data Visualization', 'protocol': 'TCP'},
        3000: {'service': 'Grafana', 'category': 'Monitoring', 'technology': 'Grafana Dashboard', 'stack': 'Metrics Visualization', 'protocol': 'TCP'},
        9090: {'service': 'Prometheus', 'category': 'Monitoring', 'technology': 'Prometheus', 'stack': 'Metrics Collection', 'protocol': 'TCP'},
        
        # System Services
        22: {'service': 'SSH', 'category': 'System', 'technology': 'Secure Shell', 'stack': 'Remote Administration', 'protocol': 'TCP'},
        21: {'service': 'FTP', 'category': 'System', 'technology': 'File Transfer Protocol', 'stack': 'File Transfer', 'protocol': 'TCP'},
        23: {'service': 'Telnet', 'category': 'System', 'technology': 'Telnet', 'stack': 'Remote Terminal', 'protocol': 'TCP'},
        53: {'service': 'DNS', 'category': 'System', 'technology': 'Domain Name Service', 'stack': 'Name Resolution', 'protocol': 'TCP/UDP'},
        25: {'service': 'SMTP', 'category': 'System', 'technology': 'Simple Mail Transfer', 'stack': 'Email Server', 'protocol': 'TCP'},
        143: {'service': 'IMAP', 'category': 'System', 'technology': 'Internet Message Access', 'stack': 'Email Server', 'protocol': 'TCP'},
        110: {'service': 'POP3', 'category': 'System', 'technology': 'Post Office Protocol', 'stack': 'Email Server', 'protocol': 'TCP'},
        
        # Container & Orchestration
        2375: {'service': 'Docker', 'category': 'Container', 'technology': 'Docker Engine', 'stack': 'Container Runtime', 'protocol': 'TCP'},
        2376: {'service': 'Docker TLS', 'category': 'Container', 'technology': 'Docker Engine TLS', 'stack': 'Secure Container Runtime', 'protocol': 'TCP'},
        6443: {'service': 'Kubernetes', 'category': 'Container', 'technology': 'Kubernetes API', 'stack': 'Container Orchestration', 'protocol': 'TCP'},
        10250: {'service': 'Kubelet', 'category': 'Container', 'technology': 'Kubernetes Kubelet', 'stack': 'Container Node Agent', 'protocol': 'TCP'},
        
        # Directory Services
        389: {'service': 'LDAP', 'category': 'Directory', 'technology': 'Lightweight Directory Access', 'stack': 'Directory Service', 'protocol': 'TCP'},
        636: {'service': 'LDAPS', 'category': 'Directory', 'technology': 'LDAP over SSL', 'stack': 'Secure Directory Service', 'protocol': 'TCP'},
        88: {'service': 'Kerberos', 'category': 'Security', 'technology': 'Kerberos Authentication', 'stack': 'Authentication Service', 'protocol': 'TCP/UDP'},
        
        # Big Data & Analytics
        8020: {'service': 'Hadoop NN', 'category': 'BigData', 'technology': 'Hadoop NameNode', 'stack': 'Distributed Storage', 'protocol': 'TCP'},
        8088: {'service': 'Hadoop RM', 'category': 'BigData', 'technology': 'Hadoop ResourceManager', 'stack': 'Cluster Resource Management', 'protocol': 'TCP'},
        7077: {'service': 'Spark', 'category': 'BigData', 'technology': 'Apache Spark', 'stack': 'Distributed Computing', 'protocol': 'TCP'},
        
        # Caching & Performance
        11211: {'service': 'Memcached', 'category': 'Cache', 'technology': 'Memcached', 'stack': 'Distributed Cache', 'protocol': 'TCP'},
        6380: {'service': 'Redis Cluster', 'category': 'Cache', 'technology': 'Redis Cluster', 'stack': 'Distributed Cache', 'protocol': 'TCP'},
    }
    
    if port_num in detailed_port_map:
        return detailed_port_map[port_num]
    else:
        return {'service': 'Unknown', 'category': 'Other', 'technology': 'Unknown Service', 'stack': 'Custom/Unknown', 'protocol': 'TCP'}

def analyze_device_tech_stack(device_name, enriched_df, device_stacks):
    """Perform comprehensive technology stack analysis for a device"""
    
    # Get all connections for this device
    device_out = enriched_df[enriched_df['initiatingDeviceName'] == device_name]
    device_in = enriched_df[enriched_df['receivingDeviceName'] == device_name]
    
    # Collect all ports
    ports_out = set()
    ports_in = set()
    
    for port in device_out['initiatingPort'].dropna():
        try:
            if str(port).isdigit():
                ports_out.add(int(port))
        except:
            continue
            
    for port in device_in['receivingPort'].dropna():
        try:
            if str(port).isdigit():
                ports_in.add(int(port))
        except:
            continue
    
    all_ports = ports_out.union(ports_in)
    
    # Analyze technologies
    technologies = []
    categories = set()
    services = []
    stacks = set()
    protocols = set()
    
    for port in all_ports:
        port_info = get_detailed_port_analysis(port)
        technologies.append(f"{port_info['technology']} (:{port})")
        categories.add(port_info['category'])
        services.append(f"{port_info['service']} (:{port})")
        stacks.add(port_info['stack'])
        protocols.add(port_info['protocol'])
    
    # Get device stack info if available
    device_stack_info = device_stacks.get(device_name, {})
    
    return {
        'detected_ports': ', '.join(map(str, sorted(all_ports))) if all_ports else 'None',
        'technology_categories': ', '.join(sorted(categories)) if categories else 'Unknown',
        'detected_technologies': ' | '.join(technologies[:5]) if technologies else 'None',  # Limit to top 5
        'detected_services': ' | '.join(services[:5]) if services else 'None',  # Limit to top 5
        'technology_stacks': ' | '.join(sorted(stacks)) if stacks else 'Unknown',
        'protocols_used': ', '.join(sorted(protocols)) if protocols else 'TCP',
        'inferred_tier': device_stack_info.get('application_tier', 'Unknown'),
        'confidence_score': device_stack_info.get('confidence_score', 0),
        'primary_role': device_stack_info.get('primary_role', 'Unknown'),
        'database_engines': ', '.join(device_stack_info.get('database_engines', [])) or 'None',
        'web_technologies': ', '.join(device_stack_info.get('web_technologies', [])) or 'None'
    }

if 'enriched_network_df' in globals() and enriched_network_df is not None:
    
    print("🔍 Analyzing technology stacks for all devices...")
    
    # Get unique devices and analyze their tech stacks
    unique_devices = set(enriched_network_df['initiatingDeviceName'].dropna()) | set(enriched_network_df['receivingDeviceName'].dropna())
    device_tech_analysis = {}
    
    for device in unique_devices:
        device_tech_analysis[device] = analyze_device_tech_stack(device, enriched_network_df, device_stacks)
    
    print(f"✅ Analyzed technology stacks for {len(device_tech_analysis)} devices")
    
    # Create enhanced export with technology analysis
    enriched_export_data = []
    
    print("🔧 Creating comprehensive enriched dataset...")
    
    for _, row in enriched_network_df.iterrows():
        # Basic connection info
        export_row = {
            'initiatingDeviceName': row.get('initiatingDeviceName', ''),
            'initiatingIpAddress': row.get('initiatingIpAddress', ''),
            'initiatingPort': row.get('initiatingPort', ''),
            'receivingDeviceName': row.get('receivingDeviceName', ''),
            'receivingIpAddress': row.get('receivingIpAddress', ''),
            'receivingPort': row.get('receivingPort', ''),
            'connectionCount': row.get('connectionCount', 0),
            'protocolName': row.get('protocolName', ''),
        }
        
        # Initiating device inventory info
        export_row.update({
            'initiating_role': row.get('roleName_initiating', 'Unknown'),
            'initiating_os': row.get('osName_initiating', 'Unknown'),
            'initiating_manufacturer': row.get('manufacturer_initiating', 'Unknown'),
            'initiating_model': row.get('model_initiating', 'Unknown'),
            'initiating_cpu': row.get('cpuType_initiating', 'Unknown'),
            'initiating_memory_mb': row.get('memoryInMb_initiating', ''),
            'initiating_is_linux': row.get('isLinux_initiating', 'Unknown'),
            'initiating_is_physical': row.get('isPhysical_initiating', 'Unknown'),
            'initiating_match_method': row.get('match_method_initiating', 'none'),
        })
        
        # Receiving device inventory info
        export_row.update({
            'receiving_role': row.get('roleName_receiving', 'Unknown'),
            'receiving_os': row.get('osName_receiving', 'Unknown'),
            'receiving_manufacturer': row.get('manufacturer_receiving', 'Unknown'),
            'receiving_model': row.get('model_receiving', 'Unknown'),
            'receiving_cpu': row.get('cpuType_receiving', 'Unknown'),
            'receiving_memory_mb': row.get('memoryInMb_receiving', ''),
            'receiving_is_linux': row.get('isLinux_receiving', 'Unknown'),
            'receiving_is_physical': row.get('isPhysical_receiving', 'Unknown'),
            'receiving_match_method': row.get('match_method_receiving', 'none'),
        })
        
        # Technology analysis for initiating device
        initiating_device = row.get('initiatingDeviceName', '')
        if initiating_device in device_tech_analysis:
            tech_info = device_tech_analysis[initiating_device]
            export_row.update({
                'initiating_detected_ports': tech_info['detected_ports'],
                'initiating_tech_categories': tech_info['technology_categories'],
                'initiating_technologies': tech_info['detected_technologies'],
                'initiating_services': tech_info['detected_services'],
                'initiating_tech_stacks': tech_info['technology_stacks'],
                'initiating_protocols': tech_info['protocols_used'],
                'initiating_inferred_tier': tech_info['inferred_tier'],
                'initiating_confidence': tech_info['confidence_score'],
                'initiating_primary_role_inferred': tech_info['primary_role'],
                'initiating_database_engines': tech_info['database_engines'],
                'initiating_web_tech': tech_info['web_technologies'],
            })
        else:
            export_row.update({
                'initiating_detected_ports': 'None',
                'initiating_tech_categories': 'Unknown',
                'initiating_technologies': 'None',
                'initiating_services': 'None',
                'initiating_tech_stacks': 'Unknown',
                'initiating_protocols': 'TCP',
                'initiating_inferred_tier': 'Unknown',
                'initiating_confidence': 0,
                'initiating_primary_role_inferred': 'Unknown',
                'initiating_database_engines': 'None',
                'initiating_web_tech': 'None',
            })
        
        # Technology analysis for receiving device
        receiving_device = row.get('receivingDeviceName', '')
        if receiving_device in device_tech_analysis:
            tech_info = device_tech_analysis[receiving_device]
            export_row.update({
                'receiving_detected_ports': tech_info['detected_ports'],
                'receiving_tech_categories': tech_info['technology_categories'],
                'receiving_technologies': tech_info['detected_technologies'],
                'receiving_services': tech_info['detected_services'],
                'receiving_tech_stacks': tech_info['technology_stacks'],
                'receiving_protocols': tech_info['protocols_used'],
                'receiving_inferred_tier': tech_info['inferred_tier'],
                'receiving_confidence': tech_info['confidence_score'],
                'receiving_primary_role_inferred': tech_info['primary_role'],
                'receiving_database_engines': tech_info['database_engines'],
                'receiving_web_tech': tech_info['web_technologies'],
            })
        else:
            export_row.update({
                'receiving_detected_ports': 'None',
                'receiving_tech_categories': 'Unknown',
                'receiving_technologies': 'None',
                'receiving_services': 'None',
                'receiving_tech_stacks': 'Unknown',
                'receiving_protocols': 'TCP',
                'receiving_inferred_tier': 'Unknown',
                'receiving_confidence': 0,
                'receiving_primary_role_inferred': 'Unknown',
                'receiving_database_engines': 'None',
                'receiving_web_tech': 'None',
            })
        
        # Port-specific analysis for this connection
        init_port = row.get('initiatingPort', '')
        recv_port = row.get('receivingPort', '')
        
        if str(init_port).isdigit():
            init_port_info = get_detailed_port_analysis(int(init_port))
            export_row.update({
                'initiating_port_service': init_port_info['service'],
                'initiating_port_category': init_port_info['category'],
                'initiating_port_technology': init_port_info['technology'],
                'initiating_port_stack': init_port_info['stack'],
            })
        else:
            export_row.update({
                'initiating_port_service': 'Unknown',
                'initiating_port_category': 'Other',
                'initiating_port_technology': 'Unknown',
                'initiating_port_stack': 'Unknown',
            })
        
        if str(recv_port).isdigit():
            recv_port_info = get_detailed_port_analysis(int(recv_port))
            export_row.update({
                'receiving_port_service': recv_port_info['service'],
                'receiving_port_category': recv_port_info['category'],
                'receiving_port_technology': recv_port_info['technology'],
                'receiving_port_stack': recv_port_info['stack'],
            })
        else:
            export_row.update({
                'receiving_port_service': 'Unknown',
                'receiving_port_category': 'Other',
                'receiving_port_technology': 'Unknown',
                'receiving_port_stack': 'Unknown',
            })
        
        enriched_export_data.append(export_row)
    
    # Create comprehensive export DataFrame
    comprehensive_export_df = pd.DataFrame(enriched_export_data)
    
    # Create Reports directory
    reports_dir = "Reports"
    if not os.path.exists(reports_dir):
        os.makedirs(reports_dir)
        print(f"📁 Created directory: {reports_dir}")
    
    # Add timestamp for internal use
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Export main comprehensive analysis with descriptive name
    main_filename = "Network_Infrastructure_Analysis_Complete.csv"
    main_filepath = os.path.join(reports_dir, main_filename)
    comprehensive_export_df.to_csv(main_filepath, index=False)
    
    print(f"✅ Successfully exported comprehensive network analysis!")
    print(f"📁 File: {main_filepath}")
    print(f"📊 Records: {len(comprehensive_export_df):,} network connections")
    print(f"📋 Columns: {len(comprehensive_export_df.columns)} data fields")
    print(f"💾 File size: {os.path.getsize(main_filepath) / 1024 / 1024:.2f} MB")
    
    print(f"\n📋 Enhanced export includes:")
    print(f"   ✅ Basic connection information (IPs, ports, protocols)")
    print(f"   ✅ Complete inventory data (hardware, OS, roles)")
    print(f"   ✅ Detailed port-based technology inference")
    print(f"   ✅ Application stack analysis")
    print(f"   ✅ Technology category classification")
    print(f"   ✅ Service identification")
    print(f"   ✅ Infrastructure tier mapping")
    print(f"   ✅ Confidence scoring")
    
    print(f"\n🔍 Column categories ({len(comprehensive_export_df.columns)} total):")
    connection_cols = [col for col in comprehensive_export_df.columns if col in ['initiatingDeviceName', 'initiatingIpAddress', 'initiatingPort', 'receivingDeviceName', 'receivingIpAddress', 'receivingPort', 'connectionCount', 'protocolName']]
    inventory_cols = [col for col in comprehensive_export_df.columns if any(x in col for x in ['_role', '_os', '_manufacturer', '_model', '_cpu', '_memory', '_linux', '_physical', '_match'])]
    tech_cols = [col for col in comprehensive_export_df.columns if any(x in col for x in ['_tech', '_service', '_stack', '_port_', '_confidence', '_tier', '_database', '_web'])]
    
    print(f"   📡 Connection data: {len(connection_cols)} columns")
    print(f"   🖥️  Inventory data: {len(inventory_cols)} columns") 
    print(f"   🔧 Technology analysis: {len(tech_cols)} columns")
    
    print(f"\n🔍 Sample of enhanced data:")
    display(comprehensive_export_df[['initiatingDeviceName', 'receivingDeviceName', 'receivingPort', 'receiving_port_technology', 'receiving_tech_categories', 'receiving_inferred_tier']].head())
    
    # Also export enhanced device summary
    if 'device_summary_df' in globals() and device_summary_df is not None:
        # Enhance device summary with detailed tech analysis
        enhanced_device_summary = []
        for _, device_row in device_summary_df.iterrows():
            device_name = device_row['Device']
            enhanced_row = device_row.to_dict()
            
            if device_name in device_tech_analysis:
                tech_info = device_tech_analysis[device_name]
                enhanced_row.update({
                    'Detected_Ports': tech_info['detected_ports'],
                    'Technology_Categories': tech_info['technology_categories'],
                    'Detected_Technologies': tech_info['detected_technologies'],
                    'Technology_Stacks': tech_info['technology_stacks'],
                    'Protocols_Used': tech_info['protocols_used'],
                })
            
            enhanced_device_summary.append(enhanced_row)
        
        enhanced_device_df = pd.DataFrame(enhanced_device_summary)
        device_summary_filename = "Device_Technology_Summary.csv"
        device_summary_filepath = os.path.join(reports_dir, device_summary_filename)
        enhanced_device_df.to_csv(device_summary_filepath, index=False)
        print(f"✅ Device technology summary exported to {device_summary_filepath}")
    
    # Export critical systems with tech details
    if 'critical_systems_df' in globals() and critical_systems_df is not None:
        critical_systems_filename = "Critical_Systems_Analysis.csv"
        critical_systems_filepath = os.path.join(reports_dir, critical_systems_filename)
        critical_systems_df.to_csv(critical_systems_filepath, index=False)
        print(f"✅ Critical systems analysis exported to {critical_systems_filepath}")
    
    # Create additional specialized reports
    print(f"\n📊 Creating specialized analysis reports...")
    
    # 1. Database Connections Report
    db_connections = comprehensive_export_df[
        comprehensive_export_df['receiving_port_category'] == 'Database'
    ].copy()
    if len(db_connections) > 0:
        db_report_filename = "Database_Connections_Report.csv"
        db_report_filepath = os.path.join(reports_dir, db_report_filename)
        db_connections.to_csv(db_report_filepath, index=False)
        print(f"✅ Database connections report exported to {db_report_filepath} ({len(db_connections)} connections)")
    
    # 2. Web Services Report
    web_connections = comprehensive_export_df[
        comprehensive_export_df['receiving_port_category'] == 'Web'
    ].copy()
    if len(web_connections) > 0:
        web_report_filename = "Web_Services_Report.csv"
        web_report_filepath = os.path.join(reports_dir, web_report_filename)
        web_connections.to_csv(web_report_filepath, index=False)
        print(f"✅ Web services report exported to {web_report_filepath} ({len(web_connections)} connections)")
    
    # 3. System Administration Report
    admin_connections = comprehensive_export_df[
        comprehensive_export_df['receiving_port_category'] == 'System'
    ].copy()
    if len(admin_connections) > 0:
        admin_report_filename = "System_Administration_Report.csv"
        admin_report_filepath = os.path.join(reports_dir, admin_report_filename)
        admin_connections.to_csv(admin_report_filepath, index=False)
        print(f"✅ System administration report exported to {admin_report_filepath} ({len(admin_connections)} connections)")
    
    # 4. Application Server Report
    app_connections = comprehensive_export_df[
        comprehensive_export_df['receiving_port_category'] == 'Application'
    ].copy()
    if len(app_connections) > 0:
        app_report_filename = "Application_Servers_Report.csv"
        app_report_filepath = os.path.join(reports_dir, app_report_filename)
        app_connections.to_csv(app_report_filepath, index=False)
        print(f"✅ Application servers report exported to {app_report_filepath} ({len(app_connections)} connections)")
    
    # 5. Security Services Report (LDAP, Kerberos, etc.)
    security_connections = comprehensive_export_df[
        (comprehensive_export_df['receiving_port_category'] == 'Directory') |
        (comprehensive_export_df['receiving_port_category'] == 'Security')
    ].copy()
    if len(security_connections) > 0:
        security_report_filename = "Security_Services_Report.csv"
        security_report_filepath = os.path.join(reports_dir, security_report_filename)
        security_connections.to_csv(security_report_filepath, index=False)
        print(f"✅ Security services report exported to {security_report_filepath} ({len(security_connections)} connections)")
    
    # 6. High-Traffic Connections Report (top 1000 by connection count)
    high_traffic = comprehensive_export_df.nlargest(1000, 'connectionCount').copy()
    if len(high_traffic) > 0:
        traffic_report_filename = "High_Traffic_Connections_Top1000.csv"
        traffic_report_filepath = os.path.join(reports_dir, traffic_report_filename)
        high_traffic.to_csv(traffic_report_filepath, index=False)
        print(f"✅ High traffic connections report exported to {traffic_report_filepath} ({len(high_traffic)} connections)")
    
    # 7. Unknown Services Report (for investigation)
    unknown_connections = comprehensive_export_df[
        (comprehensive_export_df['receiving_port_category'] == 'Other') &
        (comprehensive_export_df['receiving_port_service'] == 'Unknown')
    ].copy()
    if len(unknown_connections) > 0:
        unknown_report_filename = "Unknown_Services_Investigation.csv"
        unknown_report_filepath = os.path.join(reports_dir, unknown_report_filename)
        unknown_connections.to_csv(unknown_report_filepath, index=False)
        print(f"✅ Unknown services report exported to {unknown_report_filepath} ({len(unknown_connections)} connections)")
    
    # Create a summary report manifest
    manifest_filename = "Report_Manifest.txt"
    manifest_filepath = os.path.join(reports_dir, manifest_filename)
    
    with open(manifest_filepath, 'w') as f:
        f.write("INFRASTRUCTURE ANALYSIS REPORTS\n")
        f.write("=" * 50 + "\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Source Data: {len(comprehensive_export_df):,} network connections\n")
        f.write(f"Analyzed Devices: {len(device_tech_analysis)} unique systems\n\n")
        
        f.write("REPORT FILES:\n")
        f.write("-" * 20 + "\n")
        f.write(f"1. {main_filename}\n")
        f.write("   📋 Complete network infrastructure analysis with technology stacks\n")
        f.write(f"   📊 {len(comprehensive_export_df):,} connections, {len(comprehensive_export_df.columns)} data fields\n\n")
        
        if 'device_summary_df' in globals():
            f.write(f"2. {device_summary_filename}\n")
            f.write("   🖥️  Device-level technology summary and application stacks\n")
            f.write(f"   📊 {len(enhanced_device_df)} devices with detailed tech analysis\n\n")
        
        if 'critical_systems_df' in globals():
            f.write(f"3. {critical_systems_filename}\n")
            f.write("   🔥 Critical systems ranked by connection volume\n")
            f.write(f"   📊 Top {len(critical_systems_df)} most connected systems\n\n")
        
        report_count = 4
        if len(db_connections) > 0:
            f.write(f"{report_count}. {db_report_filename}\n")
            f.write(f"   🗄️  Database connections and technology analysis\n")
            f.write(f"   📊 {len(db_connections)} database-related connections\n\n")
            report_count += 1
        
        if len(web_connections) > 0:
            f.write(f"{report_count}. {web_report_filename}\n")
            f.write(f"   🌐 Web services and HTTP/HTTPS traffic analysis\n")
            f.write(f"   📊 {len(web_connections)} web-related connections\n\n")
            report_count += 1
        
        if len(admin_connections) > 0:
            f.write(f"{report_count}. {admin_report_filename}\n")
            f.write(f"   ⚙️  System administration and management connections\n")
            f.write(f"   📊 {len(admin_connections)} system management connections\n\n")
            report_count += 1
        
        if len(app_connections) > 0:
            f.write(f"{report_count}. {app_report_filename}\n")
            f.write(f"   📱 Application server and middleware connections\n")
            f.write(f"   📊 {len(app_connections)} application server connections\n\n")
            report_count += 1
        
        if len(security_connections) > 0:
            f.write(f"{report_count}. {security_report_filename}\n")
            f.write(f"   🔐 Security services (LDAP, Kerberos, etc.)\n")
            f.write(f"   📊 {len(security_connections)} security-related connections\n\n")
            report_count += 1
        
        if len(high_traffic) > 0:
            f.write(f"{report_count}. {traffic_report_filename}\n")
            f.write(f"   📈 High-traffic connections for performance analysis\n")
            f.write(f"   📊 Top {len(high_traffic)} highest volume connections\n\n")
            report_count += 1
        
        if len(unknown_connections) > 0:
            f.write(f"{report_count}. {unknown_report_filename}\n")
            f.write(f"   ❓ Unknown services requiring investigation\n")
            f.write(f"   📊 {len(unknown_connections)} unidentified connections\n\n")
        
        f.write("USAGE RECOMMENDATIONS:\n")
        f.write("-" * 30 + "\n")
        f.write("• Start with the Complete Analysis for overall infrastructure view\n")
        f.write("• Use specialized reports for focused analysis by technology type\n")
        f.write("• Review Critical Systems for infrastructure planning\n")
        f.write("• Investigate Unknown Services for potential security gaps\n")
        f.write("• Analyze High Traffic connections for performance optimization\n")
    
    print(f"✅ Report manifest created: {manifest_filepath}")
    
    print(f"\n📁 All reports exported to '{reports_dir}' directory!")
    print(f"📄 Total files created: {len([f for f in os.listdir(reports_dir) if f.endswith('.csv')]) + 1}")  # +1 for manifest
        
else:
    print("❌ No enriched network data available. Please run the correlation analysis first.")
    print("💡 Make sure the previous cell completed successfully.")

📤 Preparing comprehensive enriched network data for export...
🔍 Analyzing technology stacks for all devices...
✅ Analyzed technology stacks for 273 devices
🔧 Creating comprehensive enriched dataset...
✅ Analyzed technology stacks for 273 devices
🔧 Creating comprehensive enriched dataset...
✅ Successfully exported comprehensive network analysis!
📁 File: Reports/Network_Infrastructure_Analysis_Complete_20250721_192326.csv
📊 Records: 19,629 network connections
📋 Columns: 56 data fields
💾 File size: 36.09 MB

📋 Enhanced export includes:
   ✅ Basic connection information (IPs, ports, protocols)
   ✅ Complete inventory data (hardware, OS, roles)
   ✅ Detailed port-based technology inference
   ✅ Application stack analysis
   ✅ Technology category classification
   ✅ Service identification
   ✅ Infrastructure tier mapping
   ✅ Confidence scoring

🔍 Column categories (56 total):
   📡 Connection data: 8 columns
   🖥️  Inventory data: 20 columns
   🔧 Technology analysis: 24 columns

🔍 Sample of 

,initiatingDeviceName,receivingDeviceName,receivingPort,receiving_port_technology,receiving_tech_categories,receiving_inferred_tier
0,ADMERPDESA,VPROD-DOM03,445,Unknown Service,"Directory, Other, Security, System",Unknown
1,ADMERPDESA,ADMERPPROD3,139,Unknown Service,Other,Unknown
2,ADMERPDESA,VPROD-INFRA02,443,SSL/TLS Web Server,"Other, Web",Unknown
3,ADMERPDESA,VPROD-DOM01,389,Lightweight Directory Access,"Directory, Other, Security, System",Unknown
4,ADMERPDESA,DBCERT11G-RAC2,1521,Oracle Database,"Database, Other, System",Database Tier


✅ Device technology summary exported to Reports/Device_Technology_Summary_20250721_192326.csv
✅ Critical systems analysis exported to Reports/Critical_Systems_Analysis_20250721_192326.csv

📊 Creating specialized analysis reports...
✅ Database connections report exported to Reports/Database_Connections_Report_20250721_192326.csv (1788 connections)
✅ Web services report exported to Reports/Web_Services_Report_20250721_192326.csv (323 connections)
✅ System administration report exported to Reports/System_Administration_Report_20250721_192326.csv (2288 connections)
✅ Application servers report exported to Reports/Application_Servers_Report_20250721_192326.csv (31 connections)
✅ Security services report exported to Reports/Security_Services_Report_20250721_192326.csv (2170 connections)
✅ Security services report exported to Reports/Security_Services_Report_20250721_192326.csv (2170 connections)
✅ High traffic connections report exported to Reports/High_Traffic_Connections_Top1000_20250721_1

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from jaal import Jaal
import matplotlib.pyplot as plt
import os
import pandas as pd

def generate_pdf_report():
    pdf_file = os.path.join(reports_dir, "Network_Analysis_Report.pdf")
    c = canvas.Canvas(pdf_file, pagesize=letter)

    # Title
    c.setFont("Helvetica-Bold", 16)
    c.drawString(100, 750, "Network Analysis Report")
    c.setFont("Helvetica", 12)
    c.drawString(100, 730, f"Generated: {timestamp}")

    # Aggregated Data
    c.setFont("Helvetica-Bold", 13)
    c.drawString(100, 700, "Aggregated Data:")
    c.setFont("Helvetica", 12)
    c.drawString(120, 680, f"Total Devices: {len(unique_devices)}")
    c.drawString(120, 660, f"Total Connections: {len(comprehensive_export_df)}")
    c.drawString(120, 640, f"Database Connections: {len(db_connections)}")
    c.drawString(120, 620, f"Web Connections: {len(web_connections)}")
    c.drawString(120, 600, f"System Admin Connections: {len(admin_connections)}")
    c.drawString(120, 580, f"Application Server Connections: {len(app_connections)}")
    c.drawString(120, 560, f"Security Connections: {len(security_connections)}")

    # Jaal Diagram
    c.setFont("Helvetica-Bold", 13)
    c.drawString(100, 530, "Jaal Network Diagram:")
    jaal_file = os.path.join(reports_dir, "jaal_diagram.png")

    # Prepare DataFrames for Jaal
    edge_df = comprehensive_export_df[['initiatingDeviceName', 'receivingDeviceName']].dropna().copy()
    edge_df.columns = ['from', 'to']
    node_df = pd.DataFrame({'node': list(unique_devices)})

    # Generate and save Jaal diagram
    jaal_plot = Jaal(edge_df, node_df)
    jaal_plot.plot()
    plt.savefig(jaal_file)
    plt.close()

    # Insert diagram into PDF
    c.drawImage(jaal_file, 100, 300, width=400, height=200)

    c.save()
    print(f"PDF report generated: {pdf_file}")

# Run the function to generate the PDF report
generate_pdf_report()

Parsing the data...

Exception: Node dataframe missing 'id' column.

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from jaal import Jaal
import matplotlib.pyplot as plt
import os

# Function to generate PDF report with aggregated data and Jaal Diagram
def generate_pdf_report():
    pdf_file = os.path.join(reports_dir, f"Network_Analysis_Report_{timestamp}.pdf")
    c = canvas.Canvas(pdf_file, pagesize=letter)

    # Title
    c.setFont("Helvetica-Bold", 16)
    c.drawString(100, 750, "Network Analysis Report")
    c.setFont("Helvetica", 12)
    c.drawString(100, 730, f"Generated: {timestamp}")

    # Aggregated Data
    c.setFont("Helvetica-Bold", 13)
    c.drawString(100, 700, "Aggregated Data:")
    c.setFont("Helvetica", 12)
    c.drawString(120, 680, f"Total Devices: {len(unique_devices)}")
    c.drawString(120, 660, f"Total Connections: {len(comprehensive_export_df)}")
    c.drawString(120, 640, f"Database Connections: {len(db_connections)}")
    c.drawString(120, 620, f"Web Connections: {len(web_connections)}")
    c.drawString(120, 600, f"System Admin Connections: {len(admin_connections)}")
    c.drawString(120, 580, f"Application Server Connections: {len(app_connections)}")
    c.drawString(120, 560, f"Security Connections: {len(security_connections)}")

    # Jaal Diagram
    c.setFont("Helvetica-Bold", 13)
    c.drawString(100, 530, "Jaal Network Diagram:")
    jaal_file = os.path.join(reports_dir, f"jaal_diagram_{timestamp}.png")
    # Prepare data for Jaal
    jaal_edges = comprehensive_export_df[['initiatingDeviceName', 'receivingDeviceName']].dropna()
    jaal_edges.columns = ['source', 'target']
    jaal_nodes = list(unique_devices)
    jaal_data = {'nodes': [{'id': n} for n in jaal_nodes], 'edges': jaal_edges.to_dict('records')}
    # Generate and save Jaal diagram
    jaal_plot = Jaal(jaal_data)
    jaal_plot.plot()
    plt.savefig(jaal_file)
    plt.close()
    # Insert diagram into PDF
    c.drawImage(jaal_file, 100, 300, width=400, height=200)

    c.save()
    print(f"PDF report generated: {pdf_file}")

# Run the function to generate the PDF report
generate_pdf_report()